# Import & Functions

In [2]:
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import vonmises
from scipy.ndimage import gaussian_filter
from scipy.stats import gamma as gamma_dist
from scipy.special import iv as mod_bessel
from scipy.optimize import fsolve
import seaborn as sns
from scipy import optimize as opt
from scipy.signal import savgol_filter
import matplotlib as mpl
import scipy.stats as stats

# Extra libraries
import lmfit as lm
from circle_fit import taubinSVD
from circle_fit import lm as lm_fit

# Main functions

In [3]:
def clean_rep(params):
    idsrep = []
    for lb in params["Label"].unique():
        GFPcenter = params.loc[params["Label"]==lb,"GFP_Center"].values
        ids = params.loc[params["Label"]==lb,"Site_ID"]
        idf = params.loc[params["Label"]==lb,"File_ID"]   
        for c in range(len(GFPcenter)-1):
            dist = [np.linalg.norm(GFPcenter[j]-GFPcenter[c]) for j in range(c+1,len(GFPcenter))]
            dist = np.arange(c+1,c+1+len(dist))[np.array(dist)<50]        
            if len(dist) > 0:
                idsrep.append([idf.values[c],ids.values[c],ids.values[dist[0]]])
                #print("Label",lb,ids.values[c],[ids.values[c] for c in dist]," are repeated")
    for i in idsrep:
        aux = params.loc[(params["File_ID"]==i[0])&(params["Site_ID"]==i[2])]
        params = params.drop(aux.index)
    return(params)        
print("")

def p_intersect(p1,p2,u1,u2):
    t=u2[1]*(p1[0]-p2[0])-u2[0]*(p1[1]-p2[1])
    det=u1[1]*u2[0]-u1[0]*u2[1]
    if det == 0:
        p = (p1+p2)/2.
    #print(det)
    else:
        p=p1+t/det*u1
    
    return(p)
def rot_vect(v,theta):
    rot_Mat = np.array([[np.cos(theta),np.sin(theta)],[-np.sin(theta),np.cos(theta)]])
    vrot= rot_Mat.dot(v)
    return(vrot)
def PCA_center_workflow(data,do_Least_Squares):
    ## ----------------------------###
    # 1) PCA all data  --  Output: {mean_all, V_all}
    # 2) Rotated and Traslate data using V2  -- Output: {data_T}
    # 3) PCA symmetrized all data -- Output: {mean_sym,V_sym}
    # 4) Calculate 0-th center as the intersection of lines {mean_all,V_all},{mean_sym,V_sym} -- Output: {cx_sym,cy_sym}
    # If do_Least_Squares: 
    # 5) Choose subset of locs from data_T with y > yo with yo = minY+ (maxY-minY)/10*i, i=1,2,3, ... Output : {data_T_k}    
    # 6) Calculate PCA over data_T_k: --- output: {mean_k, V_k}
    # 7) Resolve the minimization problem [cx_LS,cy_LS] = min(x,y) sum_k w_k Distance(L_k,mean_k) --- Output: cx_Ls,cy_Ls
     ## ----------------------------###
    
    ######---PCA all-data ----######
    #-------------------------#
    pca = PCA(n_components=2)
    data_T = pca.fit_transform(data)
    vect = pca.components_        
    ratio= pca.explained_variance_ratio_[0]
    nv = 1
    #-------------------------#    
    
    ######---PCA symmetrized all-data ----######
    #-------------------------#
    theta = np.arctan(vect[nv,1]/vect[nv,0])
    data_T = data -np.mean(data,axis=0)
    data_T=np.apply_along_axis(rot_vect,1,data_T,theta)
    data_T[:,1]=abs(data_T[:,1])
    data_T = np.apply_along_axis(rot_vect,1,data_T,-theta)
    
    data2 = data_T + np.mean(data,axis=0)    
    data2_T = pca.fit_transform(data2)
    vect2 = pca.components_
    nv2 = 1 
    #nv2 = choose_pca_axes(data2,vect2,False)
    #-------------------------#    

    p1=np.mean(data,axis=0)
    p2=np.mean(data2,axis=0)
    p=p_intersect(p1,p2,vect[nv],vect2[nv2])  
    
    if np.linalg.norm(p1-p) > 90:
        nv2 = 0    
        p1=np.mean(data,axis=0)
        p2=np.mean(data2,axis=0)
        p=p_intersect(p1,p2,vect[nv],vect2[nv2])
        
    return(p,theta,ratio)

def calculate_PCA_params_per_site(idf,ids):    
    
    ### Apply for Experimental Data        
    ro     = 150 # nm --Max Radius to calculate radial density 
    alpha  = 1.5 # Chosen points are inside alpha*R
    do_PCA = True
    
    site_params = []
    
    #Load Data
    if grouped == True:
        data_or = np.loadtxt(home+'File_Grouped_%d_Site_%d.txt'%(idf,ids))
    else:
        data_or = np.loadtxt(home+'/File_%d_Site_%d.txt'%(idf,ids))

    nlocsG,nlocsUG = data_or[-2,:2]
    GFP_pos = data_or[-1,:2]
    locp    = data_or[:-2,2]
    data_or = data_or[:-2,:2] 
    
    n       = len(data_or)
    #print(idf,ids,len(data_or))     
    
    
    if n>30:
        
        data = np.copy(data_or)
        #print(idf,s)            

        ##-------Calculate center-------
            ## First approximation : Weighted Center of mass    
        w_i  =  (1/locp**2)
        w_i /= np.sum(w_i)
        CM_pos_Raw = np.sum((data.T*w_i).T ,axis=0)
        dist = np.linalg.norm(data-CM_pos_Raw,axis=1)
        R = np.sqrt(np.mean(dist**2))   
        R = 1.4 * R
        data = data[dist <= alpha*R]
        locp = locp[dist <= alpha*R]
        R_raw= R
            ## Second approximation : Weighted Center of mass of points inside the previous R
        if len(data)>0 :
            w_i  =  (1/locp**2)
            w_i /= np.sum(w_i)
            CM_pos = np.sum((data.T*w_i).T ,axis=0)
            dist = np.linalg.norm(data-CM_pos,axis=1)
            R = np.sqrt(np.mean(dist**2))  
            R = 1.4*R
            data = data[dist <= alpha*R]      
            locp = locp[dist <= alpha*R]
            R_2nd= R

        if len(data)>0 : 
            if do_PCA:
                    ## Third approximation : PCA_center of points inside the previous R
                    ##-------------------------   
                    [cx,cy], theta, ratio = PCA_center_workflow(data , False)               

                    dist = np.linalg.norm(data-[cx,cy],axis=1)
                    R = np.sqrt(np.mean(dist**2))        
                    R = 1.4*R
                    R_PCA = R                    
                    d_CM_PCA=np.linalg.norm(CM_pos-[cx,cy])                    
                    ##-------------------------   
            else:
                cx,cy = CM_pos
                theta = 0.0
                ratio = 0.0
                R_PCA = R_Raw
                d_CM_PCA=np.linalg.norm(CM_pos-CM_pos_Raw)                    
        else:
                cx,cy = CM_pos
                theta = 0.0
                ratio = 0.0
                R_PCA = R_Raw
                d_CM_PCA=np.linalg.norm(CM_pos-CM_pos_Raw) 
                

        ## Calculate radial_distribution        
        dist  = np.linalg.norm(data-CM_pos,axis=1)
        den   = dist                    
        data_T= np.apply_along_axis(rot_vect,1,data-CM_pos,theta)
        ang   = np.arctan2(data_T[:,1],data_T[:,0])
        
        ## Calculate Weighted-R2
        dist = np.linalg.norm(data-[cx,cy],axis=1)
        w_i  = (1/locp**2)        
        R_w  = np.sqrt(np.sum(w_i*dist**2)/np.sum(w_i)) 
        
        site_params.append([idf,ids,len(data_T),theta,ratio,R_raw,R_2nd,
                            R_PCA,R_w,den,ang,d_CM_PCA,GFP_pos[0],GFP_pos[1],CM_pos[0],CM_pos[1],cx,cy,nlocsG,nlocsUG])        
        site_params = pd.DataFrame(data=site_params,columns=["File_ID","Site_ID","N_Locs","PCA_Theta","PCA_Ratio","R_Raw","R_2nd","R_PCA",
                                                             "R_w","Radial_Coords","Angular_Coords","GFP_Distance",
                                                            "GFP_Center_X","GFP_Center_Y","Mass_Center_X","Mass_Center_Y","PCA_Center_X",
                                                             "PCA_Center_Y","Nt_Locs_Gr","Nt_Locs_UnGr"])
 
    return (site_params)

def gamma(x,a,b,c):
    return(c*x**a*np.exp(-b*x))

def fitting_per_site(site_params):  
    
    den    = site_params["Radial_Coords"].values[0]
    ang    = site_params["Angular_Coords"].values[0]
    #print(den)
    a,b,c,m_val = [1,1,1,1]
    
    gmodel = lm.Model(gamma)
        
    den,rpoints = np.histogram(den,bins=np.arange(0,150,10))        
    xx=rpoints[:-1]
    xx=xx + np.diff(xx)[0]*0.5#/max(rpoints)
    #xx=xx[:-1]+np.diff(xx)[0]/2
    n=np.sum(den)
    
    params = lm.Parameters()
    params['a'] = lm.Parameter(name='a', value=0.1*np.mean(den)+1, min=-1,vary=True)
    params['b'] = lm.Parameter(name='b', value=0.1, min=0,vary=True)
    params['c'] = lm.Parameter(name='c', value=0.1, min=0,vary=True)
        
    if n > 0:
        ## Radial fitting --> Gamma Dist.
        yy=den/n*np.diff(xx)[0]
        #print(s,xx,yy)

        result = gmodel.fit(yy, x=xx,params=params) 
        a = result.values["a"]
        b = result.values["b"]
        c = result.values["c"]
        ns= n

    
    a = a+1
    b = b+1
    c = c        
    
    site_params["Radial_Fit_Params"] = [[a,b,c]]                                        
    #site_params["Angular_Fit_Params"] = [m_val]
        
    return(site_params)

##Fix rmin calculated from scatter points, taking account the error in the localization
##--> Suppose we have an point located at a distance=rmin, then, a gaussian intensity with N(rmin,sigma)
## is calculated. The radial distribution of the intensity is given by: 
## f(r)=r/sigma**2 * Jo(r*rmin/sigma**2)*exp(-(r**2-rmin**2)/2*sigma**2) 
## For r*R/sigma**2 << 1, this can be approximated as 
## f(r)= r/sigma**2**exp(-(r**2-rmin**2)/2sigma**2 --> proportional to Rayleigh Distribution
## The CDF is given by ~   exp(-rmin**2)-exp(-(r**2-rmin**2)/2*sigma**2)
## Defining the radius minimum as CDF(ro)==epsilon
## ro = sqrt(2*sigma**2*Log(1/(1-epsilon)-R**2))
def rmin_Rayleigh_dist(rmin,epsilon,sigma):
    arg=2*sigma**2*np.log(1/(1-epsilon))-rmin**2
    if arg < 0:
        return(rmin)
    else:
        return( np.sqrt(arg))

def sinc(ang,r):
    return np.sin(ang)/ang - r

def radial_gaussian(xy,ro,sigma,amplitude,xo,yo):
    if len(np.shape(xy))!=1:
        x, y = xy    
        r = np.sqrt((x-xo)**2+(y-yo)**2)   
    else:
        r = np.abs(xy-xo)      
    #sigma = 15.
    g  = amplitude*np.exp(-(r-ro)**2/(2*sigma**2))
        
    return g.ravel()

def fitting_workflow(data):

    aux_params = data.copy()
    
    sigma=8/np.sqrt(2)
    drr=sigma // 2
    
    for j,id in enumerate(aux_params.index):
        
        idf = aux_params.loc[id,"File_ID"]
        ids = aux_params.loc[id,"Site_ID"]        
        R   = aux_params.loc[id,"R_2nd"]

        data_or = np.loadtxt(home+'File_Grouped_%d_Site_%d.txt'% (idf,ids))        
        
        GFP_cx =  aux_params.loc[id,"GFP_Center_X"]  ##GFP center        
        GFP_cy =  aux_params.loc[id,"GFP_Center_Y"]  ##GFP center        
        cx =  aux_params.loc[id,"Mass_Center_X"]         ##Mass center    
        cy =  aux_params.loc[id,"Mass_Center_Y"]         ##Mass center    
        theta =  aux_params.loc[id,"PCA_Theta"]  
        
        loc_prec= data_or[:-2,2]
        data_or = data_or[:-2,:2]  
        
        data_or = data_or -[cx,cy]                
        dist    = np.linalg.norm(data_or,axis=1)
        data_or = data_or[dist< 1.4*R]
        
        ## Taubin Fitting
        ## --------------------------------------------------------------------------------------------##        
        try:
            xc, yc, r_tau, sigma_tau = taubinSVD(data_or)
        except:
            print("Using GFP center for Tau_Center")
            xc, yc, r_tau, sigma_tau = [GFP_cx,GFP_cy,-1,-1]                                               
        
        aux_params.loc[id,"R_Tau"]      = r_tau
        aux_params.loc[id,"Sigma_Tau"]  = sigma_tau
        aux_params.loc[id,"Tau_Center_X"]  = xc
        aux_params.loc[id,"Tau_Center_Y"]  = yc
        aux_params.loc[id,"Tau_Shift"]  = np.sqrt(xc**2+yc**2)        
        ## --------------------------------------------------------------------------------------------##
        ## LM Fitting
        ## --------------------------------------------------------------------------------------------##        
        try:
            xc, yc, r_lm, sigma_lm = lm_fit(data_or,par_ini=[xc,yc,r_tau])
        except:
            print("Using GFP center for LM_Center")
            xc, yc, r_lm, sigma_lm = [GFP_cx,GFP_cy,-1,-1]                                               
        
        aux_params.loc[id,"R_LM"]      = r_lm
        aux_params.loc[id,"Sigma_LM"]  = sigma_lm
        aux_params.loc[id,"LM_Center_X"]  = xc
        aux_params.loc[id,"LM_Center_Y"]  = yc
        aux_params.loc[id,"LM_Shift"]  = np.sqrt(xc**2+yc**2)
         ## --------------------------------------------------------------------------------------------##    
        
    return(aux_params)                            


def calculate_per_site_parameters(site_params):
    
    epsilon= 0.05
    sigma  = 8
    params = [] 
    idf    = site_params["File_ID"].values
    ids    = site_params["Site_ID"].values    
    R      = site_params["R_2nd"].values[0]
    
    #Load Data
    if grouped == True:
        data_or = np.loadtxt(home+'File_Grouped_%d_Site_%d.txt'%(idf,ids))
    else:
        data_or = np.loadtxt(home+'/File_%d_Site_%d.txt'%(idf,ids))

    nlocsG,nlocsUG = data_or[-2,:2]
    GFP_pos = data_or[-1,:2]
    locp    = data_or[:-2,2]
    data_or = data_or[:-2,:2] 

    cx = site_params["Mass_Center_X"].values[0]        #Mass center
    cy = site_params["Mass_Center_Y"].values[0]        #Mass center
    GFP_cx = site_params["GFP_Center_X"].values[0] #GFP center     
    GFP_cy = site_params["GFP_Center_Y"].values[0] #GFP center    

    den     = np.linalg.norm(data_or-[cx,cy],axis=1)
    data_or = data_or[den<1.4*R]
    
    for do_center in ["Mass_Center","Tau_Center"]:
        ##-----------------------------------------------------------------------------------------------##
        fx = site_params[do_center+"_X"].values[0]            #Chosen center
        fy = site_params[do_center+"_Y"].values[0]            #Chosen centerr                                                                     

        
        if do_center in ["Fit_Center","Tau_Center","LM_Center"]:            
            fx,fy = [fx+cx,fy+cy]                
        
        if not do_center in ["Fit_Center","Tau_Center","LM_Center",
                             "Mass_Center","Mass_Center_dx","PCA_Center","GFP_Center"]:            
            print("Please,Specify the center\nUsing GFP_Center")
            fx,fy = [GFP_cx,GFP_cy]          

        den = np.linalg.norm(data_or-[fx,fy],axis=1)
        ang = np.arctan2(data_or[:,1]-fy,data_or[:,0]-fx)        
             
        site_params.at[0,"Radial_Coords"]  = den 
        site_params.at[0,"Angular_Coords"] = ang                
        
        ##-----------------------------------------------------------------------------------------------##
        ## Perform fitting ##
        site_params = fitting_per_site(site_params)
        if do_center == "Mass_Center":
            site_params = fitting_workflow(site_params)         
        ##-----------------------------------------------------------------------------------------------##


        ##-----------------------------------------------------------------------------------------------##
        ## Calculate inner_radius ##
        [alpha,b,c] = site_params["Radial_Fit_Params"].values[0]
        #alpha## This alpha value correspond to X^(a-1)*Exp(-b*x) standard form of the gamma function
        ## If we add -1 :-> alpha=radial_fit[ids,0] - 1 :-> We obtained the radial density
        b     = b-1 ## This is done because we save np.array(b_val) + 1 in order to have the same script for the betta function
        rmin  = gamma_dist.isf(1-epsilon,alpha,scale=1/b)  
        rmin_corrected=rmin_Rayleigh_dist(rmin,epsilon,sigma)
        rmin  = rmin_corrected 
        #print(do_center,alpha,b,c,[fx,fy])
        ##-----------------------------------------------------------------------------------------------##

        ##-----------------------------------------------------------------------------------------------##
        ## Calculate angular parameters ##  
        # Zn = (1/N*sum_i=1,N (z_i)) where z_i = x/sqrt(x²+y²) + iy/sqrt(x²+y²)
        # z1 = Norm(Z1): First angular moment: 
        # z2 = Norm(Z2): Second angular moment
        # Ang:  Arg{Z1}: Mean angle
        # Varx, Vary: Variance of x/sqrt(x²+y²) and y/sqrt(x²+y²) rotated-coordinates   

        x    = np.cos(ang)
        y    = np.sin(ang)     
        z    = x + y*1j
        z1   = np.mean(z)
        z2   = np.mean(z**2)
        varx = np.var(x)
        vary = np.var(y)

        dr= 15
        if do_center == "GFP_Center":
            r = site_params["Rv"].values[0]  
            s = 15            
        elif do_center == "Tau_Center":
            r = site_params["R_Tau"].values[0]  
            s = site_params["Sigma_Tau"].values[0]  
        elif do_center == "Mass_Center":            
            r = site_params["R_2nd"].values[0]/1.4  
            s = 15            
        if r < 15:
            r = r + s            
        freq,bins = np.histogram(ang[abs(den-r)<dr],bins=np.arange(-1,1+1/8,1/8)*np.pi)
        freq = savgol_filter(freq,4,3)
        fill_factor = (len(freq[freq>1.5])/len(freq))

        mean_angle  = np.angle(z1) ## Arg{z1}        
        ##-----------------------------------------------------------------------------------------------##

        ##-----------------------------------------------------------------------------------------------##
        ## Calculate copy number
        scale = [62.676,97.0055,50.56201]
        if   idf<4:
            j = 0
        elif idf<9:
            j = 1
        else:
            j = 2  
        nexos = len(x)*16/scale[j]/3
        ##-----------------------------------------------------------------------------------------------##       


        site_params["Nexos"] =  nexos
        features = ["Inner_Radius","z1","z2","varX","varY","Mean_Angle","Fill_Factor","Ratio_Var"]
        center   = do_center.split("_")[0]
        
        if do_center != "Mass_Center":
            features = [ f + "_" +center for f in features]

        site_params[features] = [rmin,np.absolute(z1),
                       np.absolute(z2),varx,vary,mean_angle,fill_factor,vary/varx]            
        
    
    site_params["Other"] = 1   
    
    return(site_params)

class processing:
    
    def __init__(self,sites):              
        self.sites  = sites
        self.nsites = nsites
        self.name   = name
        self.params = []
        self.do_ang_fit = False
        
    def run(self):        
        
        columns = ['File_ID', 'Site_ID','Label','N_Locs','R_Raw','R_2nd','R_PCA','R_w',       
       'GFP_Distance','Nt_Locs_Gr', 'PCA_Ratio','PCA_Theta', 'PCA_Center_X','PCA_Center_Y',
                   'GFP_Center_X','GFP_Center_Y','Mass_Center_X','Mass_Center_Y','Fill_Factor',
                   'Ratio_Var', "Nexos","Inner_Radius","z1","z2","varX","varY",
                  "Mean_Angle","R_Tau","Sigma_Tau","Tau_Center_X",'Tau_Center_Y',"Tau_Shift",
                 "R_LM","Sigma_LM","LM_Center_X",'LM_Center_Y',"LM_Shift",
                 "Inner_Radius_Tau","z1_Tau","z2_Tau","varX_Tau","varY_Tau",
                 "Mean_Angle_Tau","Fill_Factor_Tau","Ratio_Var_Tau"]
        
        for idf in self.sites:
            for ids in range(1,self.nsites[idf]+1):                     
                self.site_params = []
                self.site_params = calculate_PCA_params_per_site(idf,ids)
                if len(self.site_params) > 0:
                    #print(self.site_params)
                    print(idf,ids)
                    try:
                        self.site_params["Label"] = self.name[idf]
                        #self.site_params = fitting_per_site(self.site_params)
                        self.site_params = calculate_per_site_parameters(self.site_params)
                    except:
                        print("Some bug in ",idf,ids)
                        traceback.print_exc()
                        continue

                    if len(self.params) == 0:
                        self.params = self.site_params[columns]
                    else:
                        self.params = pd.concat([self.params,self.site_params[columns]],ignore_index=True)                        
                else:
                    continue           

# Running

In [7]:
# Genetics   Green   Red            Sets
# Wild_Type  Exo84   Exocyst       0,1,2,5,9,12
# Wild_Type  Sec2    Exocyst       18,19,20     
# Wild_Type  Sec9    Exocyst       15,16,17
# Sec18_Anchor_Away Exo84  Exocyst 4,8,10
# Sec9_Anchor_Away  Exo84  Exocyst 11,13,14
# Sec1_Anchor_Away  Exo84  Exocyst 6
# Wild_Type_Local Exo84  Exocyst   21
# Sec9_WT     NA      Sec9         22
# Nuclear_Pore_Complex2 NA   NA    23
# Nuclear_Pore_Complex3 NA   NA    24
# WT:18C        Exocyst   Exocyst  25
#labels  = dict({"Control":[0,1,2,9],"Sec9_AA":[11,13,14],"Sec18_AA":[4,8,10],"Sec1_AA":[6],
#           "Sec2_GFP":[18,19,20],"Sec9_GFP":[15,16,17],"WT_37C":[ 3 ],"WT_18C":[ 25 ]})

# Datatsets of interest for the paper
labels  = dict({"WT":[29,30,31,32],"Sec18_AA":[4,8,10],
                "Sec2_GFP":[18,19,20],"Sec9_GFP":[15,16,17]})
# Path to the Localization files
home     = "/home/jsortiz/phd/Paper_Tethering/github/Continuum-architecture-dynamics-of-vesicle-tethering-in-exocytosis/SMLM/Exp_Sites_Locs/"
grouped  = True  # use grouped or raw localizations
sites    = [item for sublist in labels.values() for item in sublist]#[10]#range(0,23)#[item for sublist in labels.values() for item in sublist] #range(0,23)
# number of sites for each FILE_ID
nsites   = [666,159,259,105,78,161,102,245,70,322,124,94,161,118,135,80,105,85,106,79,82,120,265,1596,379,
            52,65,30,77,20,35,90,50,300,212]
# Adding some legend
name     = ["Wild_Type_Exo84", "Wild_Type_Exo84","Wild_Type_Exo84",
            "WT_37C","Sec18","Blank","Sec1",
            "Wild_Type_Sec2","Sec18_b","Test_control","Sec18_c","Sec9","Sec15","Sec9_b","Sec9_c",
            "Exo84_Sec9","Exo84_Sec9_b","Exo84_Sec9_c","Exo84_Sec2","Exo84_Sec2_b","Exo84_Sec2_c",
            "Test_local","Sec9","NPC_2","NPC_3","WT_18C","Wild_Type_Exo84","Wild_Type_Exo84","Wild_Type_Exo84",
           "Wild_Type_Exo84","Wild_Type_Exo84","Wild_Type_Exo84","Wild_Type_Exo84","NPC_4","WT_37C"] 

workflow = processing(sites)
workflow.run()

params = workflow.params

29 1
29 2
29 4
29 5
29 7
29 8
29 9
29 10
29 13
29 14
29 15
29 17
29 18
29 20
30 1
30 4
30 5
30 8
30 9
30 11
30 13
30 15
30 17
30 18
30 20
30 21
30 23
30 25
30 27
30 28
30 29
30 30
30 31
30 32
30 33
30 34
31 1
31 2
31 3
31 4
31 5
31 6
31 8
31 9
31 11
31 12
31 13
31 15
31 16
31 17
31 19
31 20
31 24
31 26
31 27
31 28
31 29
31 30
31 31
31 32
31 33
31 34
31 35
31 36
31 37
31 38
31 39
31 41
31 42
31 45
31 47
31 48
31 49
31 50
31 51
31 53
31 54
31 57
31 58
31 59
31 60
31 61
31 62
31 64
31 65
31 66
31 67
31 68
31 69
31 70
31 71
31 73
31 74
31 75
31 76
31 77
31 78
31 79
31 80
31 81
31 82
31 83
31 84
31 85
31 86
31 87
31 88
31 89
31 90
32 3
32 4
32 5
32 6
32 7
32 8
32 9
32 10
32 11
32 13
32 14
32 15
32 17
32 18
32 20
32 21
32 22
32 23
32 24
32 26
32 27
32 28
32 30
32 32
32 33
32 34
32 35
32 36
32 40
32 41
32 42
32 43
32 44
32 45
32 47
32 48
32 49
32 50
4 1
4 2
4 3
4 4
4 5
4 6
4 7
4 8
4 9
4 10
4 11
4 12
4 14
4 15
4 16
4 17
4 18
4 19
4 20
4 21
4 22
4 23
4 24
4 26
4 27
4 30
4 32
4 33
4 34
4 35
4 36

In [ ]:
params

In [183]:
for i,l in enumerate(labels.keys()):
    params.loc[params.index.values[np.isin(params["File_ID"],labels[l])],"Set"] = l
params.loc[params["Set"]==0,"Set"] = params.loc[params["Set"]==0,"Label"] 

In [184]:
params.to_csv("All_Results_allsets.csv")